## Transformer Architecture Implementation

In this notebook, I'm going to build the standard Encoder-Decoder Transformer architecture. The goal of this notebook is to get hands-on experience with building and training the Transformer architecture.

For this task, I picked a small English-to-French translation dataset from Kaggle.

**Task:** Machine Translation

**Dataset:** https://www.kaggle.com/datasets/vishwjeetmanwar/english-french

## Dataset Preparation

In [1]:
import torch
import pandas as pd
import numpy as np

df = pd.read_csv(
    "./eng-fra.txt", sep="\t", header=None, usecols=[0, 1], names=["english", "french"]
)

df

,english,french
0,Go.,Va !
1,Go.,Marche.
2,Go.,En route !
3,Go.,Bouge !
4,Hi.,Salut !
...,...,...
239184,Death is something that we're often discourage...,La mort est une chose qu'on nous décourage sou...
239185,Since there are usually multiple websites on a...,Puisqu'il y a de multiples sites web sur chaqu...
239186,If someone who doesn't know your background sa...,Si quelqu'un qui ne connaît pas vos antécédent...
239187,It may be impossible to get a completely error...,Il est peut-être impossible d'obtenir un Corpu...


## Tokenization & Vocabulary

In [2]:
english_vocab = {"<pad>": 0, "<eos>": 1}
french_vocab = {"<pad>": 0, "<bos>": 1, "<eos>": 2}


def word_tokenize(text):
    PUNCTUATION_TO_STRIP = '.,!?"()[]{}<>:;/=_+*&^%$#@`~'

    dirty_tokens = text.lower().split()
    clean_tokens = []

    for token in dirty_tokens:
        clean_token = token.strip(PUNCTUATION_TO_STRIP)
        if clean_token:
            clean_tokens.append(clean_token)

    return clean_tokens


def forming_vocab(text, lang):
    tokenize_text = word_tokenize(text)

    if lang == "eng":
        for word in tokenize_text:
            if word not in english_vocab:
                english_vocab[word] = len(english_vocab)

    if lang == "fre":
        for word in tokenize_text:
            if word not in french_vocab:
                french_vocab[word] = len(french_vocab)


# applying
df["english"].apply(lambda text: forming_vocab(text, "eng"))
df["french"].apply(lambda text: forming_vocab(text, "fre"))

print("English vocab:", len(english_vocab))
print("French vocab:", len(french_vocab))

English vocab: 17598
French vocab: 34222


## Numericalization

In [3]:
def text_to_numbers(sent, vocab, lang):
    sent_tok = word_tokenize(sent)
    text_seq = []

    if lang == "eng":
        for word in sent_tok:
            if word in vocab:
                text_seq.append(vocab[word])

        text_seq.append(vocab["<eos>"])

    if lang == "fre":
        text_seq.append(vocab["<bos>"])

        for word in sent_tok:
            if word in vocab:
                text_seq.append(vocab[word])

        text_seq.append(vocab["<eos>"])

    return text_seq


english_sequences = df["english"].apply(
    lambda sen: text_to_numbers(sen, english_vocab, "eng")
)
french_sequences = df["french"].apply(
    lambda sen: text_to_numbers(sen, french_vocab, "fre")
)


def padding(max_len, num_sequence):
    LIST = []
    zeroes = [0] * (max_len - len(num_sequence))
    LIST = num_sequence + zeroes

    return LIST


def getMaxLength(seq):
    max_len = 0

    for s in seq.values:
        if len(s) > max_len:
            max_len = len(s)

    return max_len


print(f"Max length of English:", getMaxLength(english_sequences))
print(f"Max length of French:", getMaxLength(french_sequences))

Max length of English: 56
Max length of French: 68


## Sequence Preparation

In [4]:
from torch.utils.data import Dataset, DataLoader


class data_formation(Dataset):
    def __init__(self, english_sequences, french_sequences):
        self.english_sequences = english_sequences
        self.french_sequences = french_sequences

    def __len__(self):
        return self.english_sequences.shape[0]

    def __getitem__(self, index):
        english_sequence = padding(55, self.english_sequences[index])
        french_sequence = padding(68, self.french_sequences[index])

        return torch.tensor(english_sequence, dtype=torch.long), torch.tensor(
            french_sequence, dtype=torch.long
        )


# Train/Test Split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    english_sequences.values,
    french_sequences.values,
    test_size=0.15,
    shuffle=True,
    random_state=42,
)

training_dataset = data_formation(X_train, y_train)
testing_dataset = data_formation(X_test, y_test)

training_loader = DataLoader(training_dataset, batch_size=4)
testing_loader = DataLoader(testing_dataset, batch_size=4)

In [6]:
for x, y in training_loader:
    print(x, end="\n")
    print(y)
    break

tensor([[  91,  985, 3161,  708,   63, 2376,  902,    1,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0],
        [ 107,  249,  753,   63,    1,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0],
        [ 256,  257,   91, 1874, 1795, 8765,   66,   38,    1,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
     

## Encoder Architecture

In [16]:
# Positional Encoding matrix
import numpy as np
import torch


def get_Pencoding(
    max_seq_len: int, d_model: int, last_tok_index=None, pad: bool = False
):

    PE = np.zeros((max_seq_len, d_model))

    for pos in range(max_seq_len):
        for i in range(int(d_model / 2)):

            # pos / (10000 ** (2i/512))
            angle = pos * np.exp(-(2 * i / d_model) * np.log(10000.0))

            # get Sine
            PE[pos, 2 * i] = np.sin(angle)

            # get Cosine
            PE[pos, 2 * i + 1] = np.cos(angle)

    if pad:

        if not last_tok_index:
            raise ValueError(
                "Padding is required to identify the index of the last token in the sequence."
            )

        elif last_tok_index > max_seq_len:
            raise ValueError("last_tok_index must be <= max_seq_len")

        with_padding = np.pad(
            PE,
            ((0, max_seq_len - last_tok_index), (0, 0)),
            mode="constant",
            constant_values=0,
        )

        return with_padding

    else:
        return PE


class TransformerEmbedding:
    def __init__(self, vocab_size, max_seq_len, d_model):

        self.embedding = torch.nn.Embedding(
            num_embeddings=vocab_size, embedding_dim=d_model, padding_idx=0
        )
        self.positional_encoding = get_Pencoding(max_seq_len, d_model)

    def forward(self, x):

        emb_matrix = self.embedding(x)
        self.positional_encoding
        Pos_tensor = torch.from_numpy(self.positional_encoding).to(
            device=emb_matrix.device, dtype=emb_matrix.dtype
        )

        return emb_matrix + Pos_tensor

In [15]:
vocab_size = 17598
d_model = 512
max_seq_len = 56

input_matrix = torch.randint(0, vocab_size, (4, max_seq_len), dtype=torch.long)

encoding = TransformerEmbedding(vocab_size, max_seq_len, d_model)


encoding.forward(x=input_matrix)

tensor([[[ 0.1989,  1.0856,  0.7801,  ...,  0.8767,  0.8061,  2.5703],
         [ 1.5424,  1.5118,  2.1772,  ...,  0.2654, -0.0679, -0.2256],
         [ 0.4585, -0.6266,  0.9236,  ...,  2.3020, -0.1715,  0.2038],
         ...,
         [ 1.0097, -1.7479, -0.5352,  ...,  0.2987, -0.4632,  1.1234],
         [ 0.2128, -0.2147,  0.4559,  ..., -0.8064, -0.8817,  1.0990],
         [-1.0604, -0.6394,  0.2500,  ...,  2.5760, -0.2857,  2.1840]],

        [[ 1.6275, -1.4303,  0.1790,  ...,  0.9541,  0.2689,  0.6195],
         [ 0.6028,  0.8504, -0.1141,  ...,  1.7053,  0.1139,  1.9112],
         [-0.5018, -1.2189,  1.3151,  ...,  1.4187,  0.2940,  1.6412],
         ...,
         [ 0.7710, -1.7429,  1.3831,  ...,  1.1710, -0.3061,  1.0761],
         [ 0.9249, -1.0604,  2.4633,  ...,  0.9461,  0.5490,  1.5918],
         [-0.9879, -1.3427,  1.3409,  ...,  3.0453,  0.5332,  2.3740]],

        [[ 1.2882,  1.0744,  0.8635,  ..., -0.5189, -1.1382,  3.5570],
         [ 0.8799, -1.0932,  0.8699,  ...,  1

## Decoder Architecture

## Encoder–Decoder Model

## Training Loop

## Inference / Translation

## BLEU Evaluation